# 🧩 Mini-Lab: Notebook Smell Audit

**Module: The Python That Runs Your Pipeline** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** three common notebook smells that break reuse: cell-order dependence, hidden globals, and magic constants
2. **Explain** why each smell causes failures when a notebook is restarted, shared, or moved to production
3. **Recognize** these patterns in real code so you can flag them before they cause bugs

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Notebook vs Reusable Code | Notebook smells happen when hidden cell-order state, globals, and magic constants make code work only in one run, not when reused in a pipeline or module. |


## What Is a "Notebook Smell"?

A **code smell** is a surface-level pattern that hints at a deeper problem. In notebooks, three smells are especially dangerous because they make code **unreproducible** and **non-reusable**:

| Smell | What It Looks Like | Why It Breaks |
|-------|-------------------|---------------|
| **Cell-order dependence** | Result changes if you run cells out of order | Teammates can't reproduce your results |
| **Hidden globals** | A variable set in one cell is silently used 20 cells later | Restart kernel → `NameError` |
| **Magic constants** | Hard-coded numbers/strings with no name or explanation | Nobody knows what `0.85` means next month |

Let's see each one in action.

## Smell 1: Cell-Order Dependence

Run the two cells below **in order** — everything works. Then try running Cell B *before* Cell A after a kernel restart.

In [1]:
# --- Cell A: set up data ---
documents = [
    "LLMs generate text token by token.",
    "RAG retrieves context before generating.",
    "Embeddings capture semantic meaning.",
]
print(f"Loaded {len(documents)} documents")

Loaded 3 documents


In [2]:
# --- Cell B: process data (depends on Cell A) ---
avg_length = sum(len(d) for d in documents) / len(documents)
print(f"Average document length: {avg_length:.1f} characters")

Average document length: 36.7 characters


☝️ Cell B only works if Cell A has already run **in this kernel session**. If you restart the kernel and jump straight to Cell B, you get a `NameError` because `documents` doesn't exist yet.

**The deeper problem:** In a long notebook with 50+ cells, it's easy to accidentally run cells out of order during exploration. The notebook "works on my machine" but fails for anyone else — or even for you tomorrow.

## Smell 2: Hidden Globals

A variable is set once, then quietly reused much later — with nothing in between making the dependency obvious.

In [3]:
# Imagine this cell was run many cells ago, during exploration...
model_name = "gpt-4o-mini"
max_tokens = 500
threshold = 0.85

In [4]:
# ...then 30 cells later, this cell silently depends on all three globals
def build_request(prompt):
    return {
        "model": model_name,         # where did this come from?
        "max_tokens": max_tokens,     # set where?
        "messages": [{"role": "user", "content": prompt}],
    }

request = build_request("Explain RAG in one sentence.")
print(request)

{'model': 'gpt-4o-mini', 'max_tokens': 500, 'messages': [{'role': 'user', 'content': 'Explain RAG in one sentence.'}]}


☝️ The function *looks* like it only takes `prompt`, but it secretly depends on `model_name` and `max_tokens` defined far away. If someone imports this function into another file, those globals won't exist.

**The fix preview:** Make the function take everything it needs as parameters. We'll do exactly this in the next mini-lab.

## Smell 3: Magic Constants

Hard-coded numbers or strings with no explanation — you wrote them during a quick experiment and never named them.

In [5]:
# Smelly version — what do these numbers mean?
scores = [0.92, 0.67, 0.88, 0.45, 0.73, 0.91]

results = []
for s in scores:
    if s > 0.85:          # magic number — why 0.85?
        results.append(s)

print(f"Passing scores: {results}")
print(f"Kept {len(results)} out of {len(scores)}")

Passing scores: [0.92, 0.88, 0.91]
Kept 3 out of 6


☝️ What does `0.85` represent? A similarity threshold? A confidence cutoff? A grade boundary? Nobody knows — including future-you.

Compare with the named version:

In [6]:
# Clean version — the constant has a name
SIMILARITY_THRESHOLD = 0.85

scores = [0.92, 0.67, 0.88, 0.45, 0.73, 0.91]

results = [s for s in scores if s > SIMILARITY_THRESHOLD]

print(f"Scores above {SIMILARITY_THRESHOLD}: {results}")
print(f"Kept {len(results)} out of {len(scores)}")

Scores above 0.85: [0.92, 0.88, 0.91]
Kept 3 out of 6


Now the intent is clear, and if you need to change the threshold, there's exactly one place to do it.

## Audit Exercise: Spot the Smells

Read the notebook-style code below. It has **all three smells** baked in. The audit cell after it identifies each one.

In [7]:
# ---- Simulated "messy notebook" in a single cell ----
# (Imagine each section was a separate cell in a real notebook)

# "Cell 1" — run during morning exploration
data = ["AI is transforming industries", "Machine learning needs data", "LLMs use transformers"]
lang = "en"

# "Cell 2" — a quick experiment
cleaned = [d.lower().strip() for d in data]

# "Cell 3" — much later, uses globals from cells 1 and 2
def analyze(text_list):
    output = []
    for t in text_list:
        words = t.split()
        if len(words) > 3:             # magic constant: why 3?
            output.append({
                "text": t,
                "word_count": len(words),
                "language": lang,       # hidden global from "Cell 1"
            })
    return output

results = analyze(cleaned)             # depends on "Cell 2" global
for r in results:
    print(r)

{'text': 'ai is transforming industries', 'word_count': 4, 'language': 'en'}
{'text': 'machine learning needs data', 'word_count': 4, 'language': 'en'}


In [8]:
# ---- Audit report ----
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

audit = """
### 🔍 Audit Results

| # | Line / Area | Smell | Why It's a Problem |
|---|-------------|-------|--------------------|
| 1 | `results = analyze(cleaned)` | **Cell-order dependence** | `cleaned` only exists if "Cell 2" ran before "Cell 3" |
| 2 | `"language": lang` inside `analyze()` | **Hidden global** | `lang` is set in "Cell 1" but used inside a function — importing this function elsewhere fails |
| 3 | `if len(words) > 3` | **Magic constant** | Why 3? Is it a minimum sentence length? A filter threshold? Unnamed and unexplained |

**Score: 3 smells found** — this notebook would break on a kernel restart or if moved to a `.py` file.
"""

md(audit)


### 🔍 Audit Results

| # | Line / Area | Smell | Why It's a Problem |
|---|-------------|-------|--------------------|
| 1 | `results = analyze(cleaned)` | **Cell-order dependence** | `cleaned` only exists if "Cell 2" ran before "Cell 3" |
| 2 | `"language": lang` inside `analyze()` | **Hidden global** | `lang` is set in "Cell 1" but used inside a function — importing this function elsewhere fails |
| 3 | `if len(words) > 3` | **Magic constant** | Why 3? Is it a minimum sentence length? A filter threshold? Unnamed and unexplained |

**Score: 3 smells found** — this notebook would break on a kernel restart or if moved to a `.py` file.


## 🎯 Summary

### Key Takeaways

1. **Cell-order dependence** — if your notebook breaks when cells run top-to-bottom after a restart, you have an ordering smell
2. **Hidden globals** — variables set in one cell and silently consumed many cells later make functions non-portable and fragile
3. **Magic constants** — unnamed hard-coded values hide intent and invite bugs when thresholds need to change
4. **Smell audit habit** — before sharing or reusing notebook code, scan for these three patterns; they are the first things to fix